# Project Nord SNN + crdt-merge Integration
Run on A100. Install deps first.

In [ ]:
!pip install -q crdt-merge numpy

## 1. Federated Training — Merge checkpoints from multiple GPUs

In [ ]:
import numpy as np
from crdt_merge.model.federated import FederatedMerge
from crdt_merge.model.crdt_state import CRDTMergeState
from crdt_merge.core import GCounter, PNCounter, LWWRegister, LWWMap, ORSet
from crdt_merge.clocks import VectorClock
from crdt_merge.gossip import GossipState
import time
print('All imports OK')

In [ ]:
# Simulate Nord's 1.088B architecture (scaled down for demo)
def make_nord_state(seed, scale=1.0):
    rng = np.random.RandomState(seed)
    return {
        'sensory_zone.block_0.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'sensory_zone.block_1.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'association_zone.moe.expert_0': rng.randn(256, 256).astype(np.float32) * scale,
        'association_zone.moe.expert_1': rng.randn(256, 256).astype(np.float32) * scale,
        'memory_cortex.genesis.structural': rng.randn(96, 96).astype(np.float32) * scale,
        'memory_cortex.genesis.personal': rng.randn(96, 96).astype(np.float32) * scale,
        'memory_cortex.archive_grid': rng.randn(32, 128).astype(np.float32) * scale,
        'executive_zone.block_0.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'temporal_spike_encoder.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'lm_head.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'stdp.weight_matrix': rng.randn(64, 64).astype(np.float32) * scale,
    }

# 3 people training on different hardware
state_laptop = make_nord_state(seed=1)   # RTX 5070, 27k steps
state_colab = make_nord_state(seed=2)    # Colab T4, 15k steps  
state_cloud = make_nord_state(seed=3)    # Cloud A100, 40k steps
print(f'Each checkpoint: {sum(v.nbytes for v in state_laptop.values())/1e6:.1f} MB')

In [ ]:
# FedAvg merge — sample-weighted averaging
fed = FederatedMerge(strategy='fedavg')
fed.submit('rtx5070_laptop', state_laptop, num_samples=30000)
fed.submit('colab_t4', state_colab, num_samples=50000)
fed.submit('cloud_a100', state_cloud, num_samples=120000)

result = fed.aggregate()
print(f'Merged {result.num_clients} nodes, {result.total_samples} total samples')
print(f'Contributions: {result.client_contributions}')
print(f'Layers merged: {len(result.model)}')

## 2. CRDT Checkpoint Merge — Order-independent (the key guarantee)

In [ ]:
# Prove merge(A,B) == merge(B,A) for ANY strategy
layer = 'sensory_zone.block_0.weight'

state_ab = CRDTMergeState('weight_average')
state_ab.add(state_laptop[layer], model_id='laptop', weight=0.4)
state_ab.add(state_cloud[layer], model_id='cloud', weight=0.6)

state_ba = CRDTMergeState('weight_average')
state_ba.add(state_cloud[layer], model_id='cloud', weight=0.6)
state_ba.add(state_laptop[layer], model_id='laptop', weight=0.4)

result_ab = np.asarray(state_ab.resolve())
result_ba = np.asarray(state_ba.resolve())

print(f'Commutativity check: max diff = {np.max(np.abs(result_ab - result_ba))}')
print(f'CRDT GUARANTEE HOLDS: {np.allclose(result_ab, result_ba)}')

## 3. Sparse Delta Sync — 93% sparsity = huge bandwidth savings

In [ ]:
# Simulate Nord's 93% sparsity during training
old_weights = np.zeros((1024, 1024), dtype=np.float32)
new_weights = old_weights.copy()

# Only 7% of neurons fire (change)
rng = np.random.RandomState(42)
mask = rng.random(old_weights.shape) < 0.07
new_weights[mask] = rng.randn(mask.sum()).astype(np.float32)

# Extract sparse delta
diff = new_weights.ravel() - old_weights.ravel()
nonzero = np.abs(diff) > 1e-7
indices = np.where(nonzero)[0]
values = diff[nonzero]

full_bytes = old_weights.nbytes
sparse_bytes = indices.nbytes + values.nbytes
sparsity = 1.0 - (len(indices) / old_weights.size)

print(f'Sparsity: {sparsity:.1%}')
print(f'Full tensor: {full_bytes/1024:.0f} KB')
print(f'Sparse delta: {sparse_bytes/1024:.0f} KB')
print(f'Compression: {full_bytes/sparse_bytes:.1f}x')
print(f'\nFor 1.088B params @ 93% sparsity:')
print(f'  Full sync: {1.088e9*4/1e9:.2f} GB')
print(f'  Sparse sync: {1.088e9*0.07*12/1e9:.2f} GB')

## 4. STDP as CRDT PNCounters

In [ ]:
# STDP potentiation = increment, depression = decrement
# Maps perfectly to PNCounter (positive-negative counter)

stdp_layer0 = PNCounter()

# Node A: 1234 potentiation events, 567 depression events
stdp_layer0.increment('rtx5070', 1234)   # synapses strengthened
stdp_layer0.decrement('rtx5070', 567)    # synapses weakened

# Node B: different training run
stdp_other = PNCounter()
stdp_other.increment('a100', 800)
stdp_other.decrement('a100', 100)

# CRDT merge — order independent
merged = stdp_layer0.merge(stdp_other)
print(f'Node A net STDP: {stdp_layer0.value}')
print(f'Node B net STDP: {stdp_other.value}')
print(f'Merged net STDP: {merged.value}')
print(f'Commutativity: {stdp_layer0.merge(stdp_other).value == stdp_other.merge(stdp_layer0).value}')

## 5. Training State Tracking — Loss, Sparsity, Emergence

In [ ]:
from crdt_merge.agentic import AgentState, SharedKnowledge

# Each training node as an agent
node_a = AgentState(agent_id='rtx5070_laptop')
node_a.add_fact('loss', 4.4, confidence=1.0)
node_a.add_fact('sparsity', 0.93, confidence=1.0)
node_a.add_fact('steps', 27000, confidence=1.0)
node_a.add_tag('emergence:cross_lingual_russian')
node_a.add_tag('emergence:memory_routing_shift')
node_a.increment('spikes_observed')

node_b = AgentState(agent_id='cloud_a100')
node_b.add_fact('loss', 4.1, confidence=1.0)
node_b.add_fact('sparsity', 0.94, confidence=1.0)
node_b.add_fact('steps', 50000, confidence=1.0)
node_b.add_tag('emergence:cross_lingual_russian')
node_b.add_tag('emergence:cross_lingual_chinese')

# Merge all observations
shared = SharedKnowledge.merge(node_a, node_b)
print('=== Merged Training State ===')
print(f'Facts: {list(shared.facts.keys())}')
for k, f in shared.facts.items():
    print(f'  {k}: {f.value} (confidence={f.confidence})')
print(f'Tags: {shared.tags}')

## 6. Genesis Memory State — CRDT-replicated persistent memory

In [ ]:
# Nord's Genesis Triple Memory replicated across nodes
# Structural bank (96 neurons), Personal bank (96), Auxiliary (64)
# Plus Archive Grid with 32 RAG slots

mem_a = LWWMap()  # Node A's memory state
mem_b = LWWMap()  # Node B's memory state
ts = time.time()

# Node A learns structural patterns
mem_a.set('structural_0', 0.95, timestamp=ts, node_id='node_a')
mem_a.set('structural_1', 0.82, timestamp=ts, node_id='node_a')
mem_a.set('archive_slot_0', 'subject-verb-object', timestamp=ts, node_id='node_a')

# Node B learns different patterns
mem_b.set('structural_0', 0.91, timestamp=ts+1, node_id='node_b')  # newer
mem_b.set('structural_2', 0.77, timestamp=ts, node_id='node_b')
mem_b.set('archive_slot_1', 'noun-phrase-structure', timestamp=ts, node_id='node_b')

# CRDT merge — LWW per key
merged_mem = mem_a.merge(mem_b)
print('Merged memory state:')
for k, v in sorted(merged_mem.value.items()):
    print(f'  {k}: {v}')
print(f'\nstructural_0 = {merged_mem.value["structural_0"]} (node_b wins — newer timestamp)')

## 7. Gossip Protocol — Peer-to-peer sync without central server

In [ ]:
# No parameter server needed. Each node gossips state to peers.
node1 = GossipState('laptop', fanout=2)
node2 = GossipState('colab', fanout=2)
node3 = GossipState('cloud', fanout=2)

# Each node records its training progress
node1.update('loss', 4.4)
node1.update('step', 27000)

node2.update('loss', 4.6)
node2.update('step', 15000)

node3.update('loss', 4.1)
node3.update('step', 50000)

# Anti-entropy: node1 compares digest with node2
digest1 = node1.digest()
digest2 = node2.digest()
push, pull = node1.anti_entropy_push_pull(digest2)
print(f'Node1 should push {len(push)} keys to Node2')
print(f'Node1 should pull {len(pull)} keys from Node2')

# Full merge
merged = node1.merge(node2).merge(node3)
print(f'\nMerged gossip state: {merged.size} live entries')
for key in ['loss', 'step']:
    val = merged.get(key)
    print(f'  {key}: {val}')

## 8. Vector Clocks — Causal ordering of training events

In [ ]:
from crdt_merge.clocks import Ordering

# Track causal ordering: which training events happened-before others
clock_a = VectorClock({'laptop': 27000, 'colab': 0})
clock_b = VectorClock({'laptop': 0, 'colab': 15000})
clock_c = VectorClock({'laptop': 5000, 'cloud': 50000})

print(f'A vs B: {clock_a.compare(clock_b)}')  # CONCURRENT
print(f'A vs C: {clock_a.compare(clock_c)}')  # CONCURRENT

merged_clock = clock_a.merge(clock_b).merge(clock_c)
print(f'\nMerged clock: {dict(merged_clock._clocks)}')
print('All events are now causally ordered.')

## 9. Sparse-Aware Merge -- Why naive averaging kills SNNs

In [ ]:
# The problem: naive averaging DESTROYS sparse spike signals
# Neuron fires (0.8) on node A, silent (0.0) on node B
# Naive avg: (0.8+0.0)/2 = 0.4 -- signal halved, spike pattern lost
# OR-Set merge: non-zero wins over zero, signal preserved

def sparse_aware_merge(state_dicts, sparsity_threshold=0.01):
    """OR-Set inspired merge: non-zero weights win over zeros."""
    merged = {}
    all_keys = set()
    for sd in state_dicts:
        all_keys.update(sd.keys())
    for key in all_keys:
        tensors = [np.asarray(sd[key], dtype=np.float32)
                   for sd in state_dicts if key in sd]
        if len(tensors) == 1:
            merged[key] = tensors[0]
            continue
        shape = tensors[0].shape
        active_sum = np.zeros(shape, dtype=np.float64)
        active_count = np.zeros(shape, dtype=np.float64)
        for t in tensors:
            mask = np.abs(t) > sparsity_threshold
            active_sum += np.where(mask, t.astype(np.float64), 0.0)
            active_count += mask.astype(np.float64)
        result = np.zeros(shape, dtype=np.float32)
        nonzero_mask = active_count > 0
        result[nonzero_mask] = (active_sum[nonzero_mask] / active_count[nonzero_mask]).astype(np.float32)
        merged[key] = result
    return merged

# Demonstrate the difference
firing = np.array([0.8, 0.0, 0.6, 0.0, 0.9], dtype=np.float32)
silent = np.array([0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)

naive_avg = (firing + silent) / 2
sparse_result = sparse_aware_merge([{'w': firing}, {'w': silent}])['w']

print('Naive averaging vs Sparse-aware merge:')
for i in range(len(firing)):
    status = 'PRESERVED' if abs(sparse_result[i] - firing[i]) < 0.01 else 'LOST'
    print(f'  Neuron {i}: input={firing[i]:.1f} | naive={naive_avg[i]:.1f} | sparse={sparse_result[i]:.1f} [{status}]')

# Now test with two 93% sparse tensors
w1 = np.zeros((1000, 1000), dtype=np.float32)
w2 = np.zeros((1000, 1000), dtype=np.float32)
rng1, rng2 = np.random.RandomState(1), np.random.RandomState(2)
m1 = rng1.random((1000, 1000)) < 0.07
m2 = rng2.random((1000, 1000)) < 0.07
w1[m1] = rng1.randn(m1.sum()).astype(np.float32)
w2[m2] = rng2.randn(m2.sum()).astype(np.float32)

naive = (w1 + w2) / 2
sparse = sparse_aware_merge([{'w': w1}, {'w': w2}])['w']

naive_sparsity = (np.abs(naive) < 0.01).sum() / naive.size
sparse_sparsity = (np.abs(sparse) < 0.01).sum() / sparse.size

print(f'\n93% sparse tensors merged:')
print(f'  Naive avg sparsity:  {naive_sparsity:.1%} (signal diluted)')
print(f'  Sparse merge sparsity: {sparse_sparsity:.1%} (structure preserved)')

## 10. Shard Training on Free Tiers -- Scale horizontally for $0

In [ ]:
# Split model into shards, train each on free platforms, merge back
full_state = make_nord_state(seed=0)
layers = list(full_state.keys())
num_shards = 4
shard_size = len(layers) // num_shards
shards = {}
for i in range(num_shards):
    start = i * shard_size
    end = start + shard_size if i < num_shards - 1 else len(layers)
    shards[f'shard_{i}'] = {k: full_state[k].copy() for k in layers[start:end]}

platforms = ['Colab Free', 'Kaggle Free', 'Local RTX 5070', 'Colab Free #2']
trained_shards = {}
for i, (sid, shard) in enumerate(shards.items()):
    rng_t = np.random.RandomState(i + 100)
    trained_shards[sid] = {k: v + rng_t.randn(*v.shape).astype(np.float32)*0.01
                           for k, v in shard.items()}
    print(f'{sid} ({platforms[i]}): {len(shard)} layers, {sum(v.size for v in shard.values()):,} params, cost: $0.00')

# Merge all shards using sparse-aware merge
merged_model = {}
for key in set(k for sd in trained_shards.values() for k in sd):
    parts = [sd[key] for sd in trained_shards.values() if key in sd]
    merged_model[key] = parts[0] if len(parts) == 1 else sparse_aware_merge([{key: p} for p in parts])[key]

print(f'\nMerged: {len(merged_model)} layers, {sum(v.size for v in merged_model.values()):,} params')
print(f'Total cost: $0.00 | Equivalent single-run: ~$670+')


## 11. Surrogate Gradient Health Monitoring

In [ ]:
# Track surrogate gradient health across distributed training nodes
# Vanishing gradients are the #1 failure mode for SNN training at scale

grad_node_a = LWWMap()
grad_node_b = LWWMap()
vanish_a = GCounter()
vanish_b = GCounter()
explode_a = GCounter()
grad_steps_a = GCounter()
grad_steps_b = GCounter()

# Node A: mostly healthy gradients, one vanishing event
for layer in ['sensory.block_0', 'association.moe', 'memory_cortex', 'executive.block_0']:
    norm = np.random.uniform(0.01, 0.1)
    grad_node_a.set(layer, norm, timestamp=time.time(), node_id='rtx5070')
    grad_steps_a.increment('rtx5070')
    if norm < 1e-7:
        vanish_a.increment('rtx5070')

# Simulate a vanishing gradient event
grad_node_a.set('deep_stdp_layer', 1e-9, timestamp=time.time(), node_id='rtx5070')
vanish_a.increment('rtx5070')
grad_steps_a.increment('rtx5070')

# Node B: one exploding gradient
for layer in ['sensory.block_0', 'association.moe', 'executive.block_0']:
    grad_node_b.set(layer, np.random.uniform(0.02, 0.08), timestamp=time.time(), node_id='a100')
    grad_steps_b.increment('a100')

grad_node_b.set('lm_head', 500.0, timestamp=time.time(), node_id='a100')
explode_a.increment('a100')
grad_steps_b.increment('a100')

# Merge gradient stats from both nodes
merged_norms = grad_node_a.merge(grad_node_b)
merged_vanish = vanish_a.merge(vanish_b)
merged_explode = explode_a.merge(GCounter())
merged_steps = grad_steps_a.merge(grad_steps_b)

print('=== Surrogate Gradient Health (merged from 2 nodes) ===')
print(f'Total gradient steps: {merged_steps.value}')
print(f'Vanishing events: {merged_vanish.value}')
print(f'Exploding events: {merged_explode.value}')
print(f'\nPer-layer gradient norms (latest from any node):')
for layer, norm in sorted(merged_norms.value.items()):
    status = 'VANISHING' if norm < 1e-7 else ('EXPLODING' if norm > 100 else 'ok')
    print(f'  {layer}: {norm:.6f} [{status}]')

vanish_rate = merged_vanish.value / max(merged_steps.value, 1)
print(f'\nHealth: {"HEALTHY" if vanish_rate < 0.1 else "WARNING: high vanishing rate"} ({vanish_rate:.0%} vanishing)')

## Summary

| Nord Problem | crdt-merge Solution | CRDT Type |
|---|---|---|
| Budget exhaustion ($670) | Federated training across donated GPUs | FederatedMerge |
| 93% sparsity | Sparse delta sync (4-5x bandwidth savings) | GossipState + delta encoding |
| STDP weight updates | Potentiation/depression counters | PNCounter |
| Memory cortex (39% activation) | Replicated persistent memory | LWWMap |
| Cross-lingual emergence | Capability tracking (add-wins) | ORSet |
| Checkpoint merging | Order-independent merge | CRDTMergeState |
| No central server | Peer-to-peer gossip | GossipState |
| Causal ordering | Distributed clocks | VectorClock |